# TP2 — Baseline Logistic Regression & Métriques
## Fil rouge : Churn Predictor (Telco Customer Churn)

**Durée estimée : 1h15**

### 🎯 Objectifs d'apprentissage
À la fin de ce TP vous saurez :
- Construire un **`ColumnTransformer`** propre (numérique + catégoriel) intégré dans un `Pipeline` scikit-learn.
- Entraîner une **régression logistique** baseline et la comparer à un `DummyClassifier`.
- Lire les **métriques de classification** : accuracy, precision, recall, F1, ROC-AUC, PR-AUC.
- Comprendre pourquoi l'**accuracy ment** sur un dataset déséquilibré.
- **Ajuster le seuil de décision** selon un objectif métier.
- Lire les **coefficients** d'une régression logistique pour interpréter le modèle.

### 📦 Pré-requis
- Avoir terminé le TP1 (les splits doivent être dans `./data/`).

## 0. Setup & rechargement des splits

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
    roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve,
)

sns.set_theme(style="whitegrid")


In [ ]:
X_train = pd.read_csv("data/X_train.csv")
X_val   = pd.read_csv("data/X_val.csv")
y_train = pd.read_csv("data/y_train.csv")["Churn"]
y_val   = pd.read_csv("data/y_val.csv")["Churn"]

print(f"X_train: {X_train.shape} | X_val: {X_val.shape}")


## 1. Identifier les colonnes numériques et catégorielles

On va appliquer des **traitements différents** :
- Numériques → imputation médiane + standardisation.
- Catégorielles → imputation mode + one-hot encoding.

🧠 **Réflexe Tech Lead** : ne JAMAIS hardcoder la liste des colonnes. On la dérive dynamiquement de `X_train` pour que le code reste valide si le schéma évolue.

In [ ]:
# TODO: Identify numeric columns (int, float) using select_dtypes
num_cols = ...

# TODO: Identify categorical columns (object) using select_dtypes
cat_cols = ...

print(f"Numeric ({len(num_cols)}): {num_cols}")
print(f"Categorical ({len(cat_cols)}): {cat_cols}")


## 2. Construction du `ColumnTransformer`

`ColumnTransformer` applique des transformations différentes à différentes colonnes, en parallèle, dans un seul objet sklearn. C'est la **brique fondamentale** d'un pipeline ML production-ready.

🎯 **Tâche** : construisez deux sous-pipelines (numérique et catégoriel), puis assemblez-les dans un `ColumnTransformer` nommé `preprocessor`.

⚠️ **Point de vigilance** : utilisez `handle_unknown="ignore"` sur le `OneHotEncoder`. Sinon, si une catégorie inédite apparaît en val/test/prod, le modèle plante.

In [ ]:
# TODO: Build the numeric sub-pipeline:
#   1) SimpleImputer(strategy="median")
#   2) StandardScaler()
numeric_pipe = ...

# TODO: Build the categorical sub-pipeline:
#   1) SimpleImputer(strategy="most_frequent")
#   2) OneHotEncoder(handle_unknown="ignore")
categorical_pipe = ...

# TODO: Combine them in a ColumnTransformer
preprocessor = ...

preprocessor


## 3. Le baseline qu'il faut battre : `DummyClassifier`

Avant tout modèle "intelligent", on entraîne un **classifieur trivial** qui prédit toujours la classe majoritaire (`No churn`). C'est le **plancher** : tout modèle qui ne fait pas mieux que ça est inutile.

🧠 **Pourquoi c'est un réflexe Tech Lead crucial** : sur un dataset à 73 % de "No churn", le DummyClassifier fait **73 % d'accuracy** sans rien apprendre. Si vous communiquez à un PM "mon modèle fait 75 % d'accuracy", il pensera que c'est bien — mais ça ne fait que **2 points de mieux qu'un coup de chance**.

In [ ]:
dummy = Pipeline([
    ("prep", preprocessor),
    ("clf", DummyClassifier(strategy="most_frequent")),
])
dummy.fit(X_train, y_train)

dummy_acc = dummy.score(X_val, y_val)
print(f"Dummy accuracy on val: {dummy_acc:.4f}")
print(f"(This is the floor any real model must beat by a wide margin.)")


## 4. Premier modèle : régression logistique

La régression logistique est **le baseline standard** en classification :
- Rapide à entraîner.
- Interprétable (chaque feature a un coefficient).
- Excellente référence pour juger les modèles plus complexes.

In [ ]:
# TODO: Build a Pipeline with:
#   - the preprocessor we built earlier
#   - LogisticRegression(max_iter=1000, random_state=42)
model = ...

# TODO: Fit on train, predict on val (both class and probabilities)
# - y_pred  = predicted classes (0/1)
# - y_proba = predicted probability of churn (class 1)
y_pred = ...
y_proba = ...


## 5. Lire les métriques

On va calculer les métriques classiques **et comprendre ce que chacune nous dit** :

| Métrique | Question répondue |
|----------|-------------------|
| Accuracy | "Quel % des prédictions sont correctes ?" — trompeuse si déséquilibré. |
| Precision | "Quand je prédis churn, combien de fois j'ai raison ?" |
| Recall | "Parmi les vrais churners, combien j'en attrape ?" |
| F1 | Moyenne harmonique de precision et recall. |
| ROC-AUC | Capacité du modèle à **classer** un churner au-dessus d'un non-churner. |
| PR-AUC | Comme ROC-AUC mais plus pertinent en **classes déséquilibrées**. |

In [ ]:
# TODO: Compute and print the 6 metrics: accuracy, precision, recall, F1, ROC-AUC, PR-AUC


# TODO: Print the full classification report (sklearn.metrics.classification_report)


💡 **Lecture critique** :
- L'accuracy (~0.81) est seulement ~8 points au-dessus du dummy (0.73). Pas si impressionnant.
- Le **recall sur la classe Churn** (~0.55) est le vrai sujet : on **rate ~45 %** des churners.
- C'est cohérent avec le déséquilibre : le modèle est biaisé vers la classe majoritaire.

➡️ Conséquence métier : si l'objectif est "ne pas laisser partir les churners sans tenter une rétention", il faut **augmenter le recall**, quitte à sacrifier de la precision. C'est un arbitrage à porter au métier — pas une décision technique unilatérale.

## 6. Matrice de confusion

La matrice de confusion donne la vue exhaustive : combien de TP, FP, TN, FN.

In [ ]:
cm = confusion_matrix(y_val, y_pred)

plt.figure(figsize=(5, 4))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=["No churn", "Churn"],
    yticklabels=["No churn", "Churn"],
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion matrix @ threshold = 0.5")
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"True Negatives:  {tn}")
print(f"False Positives: {fp}  (we annoy {fp} happy clients)")
print(f"False Negatives: {fn}  (we miss {fn} actual churners — costly)")
print(f"True Positives:  {tp}  (we correctly catch {tp} churners)")


## 7. Courbes ROC et Precision-Recall

- **ROC** : trade-off entre TPR (recall) et FPR. Bonne pour des classes équilibrées.
- **PR** : trade-off entre Precision et Recall. **Bien plus informative** quand la classe positive est rare.

Sur un dataset à 26 % de churn, regardez les **deux**, mais privilégiez la **PR-AUC** pour communiquer.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# ROC curve
fpr, tpr, _ = roc_curve(y_val, y_proba)
axes[0].plot(fpr, tpr, label=f"AUC = {roc_auc_score(y_val, y_proba):.3f}")
axes[0].plot([0, 1], [0, 1], "--", color="gray", label="Random")
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].set_title("ROC curve")
axes[0].legend()

# Precision-Recall curve
prec, rec, _ = precision_recall_curve(y_val, y_proba)
axes[1].plot(rec, prec, label=f"AP = {average_precision_score(y_val, y_proba):.3f}")
axes[1].axhline(y=y_val.mean(), color="gray", linestyle="--",
                label=f"Baseline = {y_val.mean():.2f}")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-Recall curve")
axes[1].legend()

plt.tight_layout()
plt.show()


## 8. Ajustement du seuil de décision

Par défaut, `predict()` utilise un seuil de **0.5**. Mais ce n'est qu'une convention, **pas un choix métier**.

🎯 **Tâche** : essayez plusieurs seuils (0.5, 0.4, 0.35, 0.3, 0.25) et observez l'évolution precision/recall/F1.

🧠 **Question Tech Lead** : pour notre cas churn, quel seuil privilégier ?
- Si chaque appel de rétention coûte 5 € et qu'un churn évité rapporte 200 €, on accepte de **gros faux positifs** pour gagner du recall.
- Le bon seuil dépend de l'**économie du problème**, pas du modèle.

In [ ]:
# TODO: For each threshold in [0.5, 0.4, 0.35, 0.3, 0.25]:
#   1) compute y_pred_t = (y_proba >= threshold)
#   2) print precision, recall, f1
# Identify which threshold maximizes F1.

thresholds = [0.5, 0.4, 0.35, 0.3, 0.25]


## 9. Lire les coefficients du modèle

La régression logistique est **interprétable** : chaque feature a un coefficient.
- Coefficient **positif** → augmente la probabilité de churn.
- Coefficient **négatif** → la diminue.
- L'**amplitude** (en valeur absolue) indique l'importance.

⚠️ **Attention** : ces coefficients sont sur les features **standardisées et one-hot encodées**. On lit `cat__Contract_Month-to-month` pas juste `Contract`.

In [ ]:
# TODO:
# 1) Get the feature names after preprocessing using model.named_steps["prep"].get_feature_names_out()
# 2) Get the coefficients from model.named_steps["clf"].coef_[0]
# 3) Build a DataFrame {feature, coef}, sort by absolute value descending
# 4) Print the top 15 features
# 5) Plot a horizontal bar chart of the top 15


💡 **Lecture attendue** :
- `Contract_Month-to-month`, `InternetService_Fiber optic`, `PaymentMethod_Electronic check` ressortent en positif → cohérent avec l'EDA.
- `Contract_Two year`, `tenure` ressortent en négatif → ancienneté et engagement protègent.

🧠 **Réflexe Tech Lead** : un modèle interprétable est un **outil de communication métier**. Vous pouvez aller voir le PM avec : "Voilà les 5 leviers que le modèle identifie. Lesquels êtes-vous capable d'actionner business ?". C'est ça la différence entre Data Scientist et Tech Lead.

## 10. Synthèse : votre baseline

🎯 **À remplir** :

> **Baseline retenue** : régression logistique avec preprocessing standard
>
> **Score val (seuil 0.5)** :
> - Accuracy : ...
> - Precision : ...
> - Recall : ...
> - F1 : ...
> - PR-AUC : ...
>
> **Top 3 features identifiées** :
> 1. ...
> 2. ...
> 3. ...
>
> **Limites identifiées** :
> - ...
>
> **Pistes pour la session 2** :
> - Tester arbres / gradient boosting (capture de non-linéarités)
> - Gérer le déséquilibre (class_weight, SMOTE)
> - Cross-validation au lieu d'un seul split val
> - GridSearch sur les hyperparamètres
> - Tracker les expériences avec MLflow

## 🚀 Bonus

1. **Régularisation** : entraînez `LogisticRegression(C=0.01)` puis `C=100`. Lequel performe mieux ? Pourquoi ?
2. **Class weight** : ajoutez `class_weight="balanced"` à la régression logistique. Comment évoluent precision et recall ? Cohérent avec votre intuition ?
3. **Optimal threshold** : trouvez le seuil qui maximise le F1 par recherche exhaustive (par exemple sur 100 valeurs entre 0 et 1). Comparez avec le seuil 0.5.
4. **Calibration** : tracez la `calibration_curve` de scikit-learn. Le modèle est-il bien calibré (les probabilités prédites correspondent-elles aux fréquences réelles) ?